# Reading Wikipedia Articles from Kaggle

This notebook demonstrates how to read Wikipedia articles using datasets available on Kaggle.

**Motivation:**

*   Wikipedia articles have been uploaded to Kaggle to provide developers with easy access, eliminating the need to scrape Wikipedia directly.

**Process:**

1.  **Download the Index:** We first download a Wikimedia index/dictionary file. This file helps locate specific articles within the larger dataset structure.
2.  **Download Specific Articles:** Using the index, we then download only the particular articles relevant to our needs.


In [36]:
import kagglehub
import os
import json
from tqdm import tqdm
import pandas as pd
from fuzzywuzzy import fuzz, process

In [5]:
def read_jsonl(file_path, max_records=None):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(tqdm(f)):
            if max_records and i >= max_records:
                break
            data.append(json.loads(line))
    return data

In [ ]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

dataset_slug = "mehranism/wikimedia-structured-dataset-navigator-jsonl"
dataset_files = api.dataset_list_files(dataset_slug)

print("Files in the dataset:")
for f in dataset_files.files:
    print(f.name)

Files in the dataset:
wiki_structured_dataset_navigator.jsonl


In [27]:
import kagglehub

dataset_slug = "mehranism/wikimedia-structured-dataset-navigator-jsonl"
file_name = "wiki_structured_dataset_navigator.jsonl"  # Replace with the actual file name

downloaded_file_path = kagglehub.dataset_download(dataset_slug, path=file_name)
print(f"File downloaded to: {downloaded_file_path}")

100%|██████████| 1.38G/1.38G [00:42<00:00, 34.6MB/s]

File downloaded to: /Users/aris/.cache/kagglehub/datasets/mehranism/wikimedia-structured-dataset-navigator-jsonl/versions/1/wiki_structured_dataset_navigator.jsonl


In [31]:
def load_jsonl(file_path, max_records=None):
    """Load a JSONL file into a list of dictionaries.
    
    Args:
        file_path (str): Path to the JSONL file.
        max_records (int, optional): Maximum number of records to load.
    
    Returns:
        list: List of parsed JSON objects.
    """
    records = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for i, line in enumerate(tqdm(file, desc="Loading JSONL")):
            if max_records and i >= max_records:
                break
            records.append(json.loads(line))
    return records

In [33]:
# Load the dataset
records = load_jsonl(downloaded_file_path)

# Convert to DataFrame and free memory
navigator_df = pd.DataFrame(records)
del records
navigator_df.head(3)

Loading JSONL: 7405030it [00:31, 233854.71it/s]


,file_name,file_index,name,url,description
0,enwiki_namespace_0_0.jsonl,0,Not Again SU,https://en.wikipedia.org/wiki/Not_Again_SU,"Student organization in Syracuse, New York"
1,enwiki_namespace_0_0.jsonl,1,Mike Ross (Suits),https://en.wikipedia.org/wiki/Mike_Ross_(Suits),An American Legal drama series from 2011-2019
2,enwiki_namespace_0_0.jsonl,2,Second Polish republic,https://en.wikipedia.org/wiki/Second_Polish_re...,Country in Central and Eastern Europe (1918–1939)


In [37]:
def fuzzy_article_search(query, dataframe, column_name, scorer=fuzz.token_sort_ratio, limit=5):
    """Perform fuzzy matching to find articles matching a query.
    
    Args:
        query (str): User's search query.
        dataframe (pd.DataFrame): DataFrame containing article metadata.
        column_name (str): Column to search (e.g., 'name').
        scorer (function): Fuzzy matching scorer function.
        limit (int): Number of top matches to return.
    
    Returns:
        list: List of dictionaries with index, name, and score for matches.
    """
    # Extract article names
    article_names = dataframe[column_name].tolist()
    
    # Perform fuzzy matching
    matches = process.extract(query, article_names, scorer=scorer, limit=limit)
    
    # Compile results with DataFrame indices
    results = []
    for match in matches:
        name, score = match[0], match[1]
        index = dataframe[dataframe[column_name] == name].index[0]
        results.append({"index": index, "name": name, "score": score})
    
    return results

In [ ]:
# Example query
user_query = "Shawshenk Redemption"

# Run fuzzy search
search_results = fuzzy_article_search(user_query, navigator_df, "name", scorer=fuzz.token_sort_ratio, limit=5)

# Display results
print("\nFuzzy search results:")
for result in search_results:
    print(f"Index: {result['index']}, Name: {result['name']}, Score: {result['score']}")

KeyboardInterrupt: 

KeyboardInterrupt: 

In [ ]:
def fetch_article_content(file_name, file_index):
    """Retrieve article data using file name and index.
    
    Args:
        file_name (str): Name of the file containing the article.
        file_index (int): Index of the article in the file.
    
    Returns:
        pd.DataFrame: DataFrame with the article's data.
    """
    # Download the article file
    file_path = kagglehub.dataset_download(
        "wikimedia-foundation/wikipedia-structured-contents",
        path=f"enwiki_namespace_0/{file_name}"
    )
    
    # Load the JSONL file
    article_records = load_jsonl(file_path)
    
    # Convert to DataFrame and free memory
    article_df = pd.DataFrame(article_records)
    del article_records
    
    # Return the specific article
    return article_df[article_df.index == file_index]

In [ ]:
# Select the top search result
top_result_index = search_results[0]["index"]
article_file_name = navigator_df.iloc[top_result_index].file_name
article_file_index = navigator_df.iloc[top_result_index].file_index

# Fetch and display the article data
print("\nFull article data for top match:")
article_content = fetch_article_content(article_file_name, article_file_index)
display(article_content)